# Neural Network Fundamentals Solutions

Solutions for `exercises.ipynb`. Try the exercises first — peek here only after you've attempted each question.

## Part 1 — Warm-up

**1. Your first neuron.** `np.dot(w, x) + b` is the neuron's total evidence; the step threshold turns evidence into a fire/no-fire decision.

In [ ]:
import numpy as np

w = np.array([0.6, 0.9, 0.4])
b = -3.0

def step(z):
    return 1 if z > 0 else 0

for name, x in [("Lovely evening", np.array([7.0, 8.0, 6.0])),
                ("Rainy tired day", np.array([3.0, 2.0, 5.0]))]:
    z = np.dot(w, x) + b
    print(f"{name:18s} z = {z:5.2f} -> fires? {bool(step(z))}")

**2. Two activations.** Sigmoid squashes everything into (0, 1); ReLU keeps positives and zeroes negatives - and its slope never vanishes there.

In [ ]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def relu(z):
    return np.maximum(0, z)

z = np.array([-2.0, -0.5, 0.0, 1.5])
print("sigmoid:", sigmoid(z).round(4))
print("relu  :", relu(z))
# tanh-style activations reach negative values; sigmoid never does.

**3. Score your guesses with MSE.** MSE squares each error before averaging, so large misses are punished brutally.

In [ ]:
import numpy as np

def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

rent = np.array([20.0, 35.0])
pred_good = np.array([21.0, 34.0])
pred_bad = np.array([30.0, 45.0])

print(f"good model MSE: {mse(rent, pred_good):.2f}")
print(f"bad model  MSE: {mse(rent, pred_bad):.2f}")

## Part 2 — Practice

**4. Forward pass of a 2-4-1 network.** Each column of W1 is one hidden neuron; one `@` evaluates all of them for all samples, and the (1, 4) bias broadcasts down every row.

In [ ]:
import numpy as np

rng = np.random.default_rng(42)
W1 = rng.normal(0, 0.5, size=(2, 4))
b1 = np.zeros((1, 4))
W2 = rng.normal(0, 0.5, size=(4, 1))
b2 = np.zeros((1, 1))
X = np.array([[0.5, 1.5],
              [2.0, 0.5],
              [1.0, 1.0],
              [3.0, 2.5]])

Z1 = X @ W1 + b1
A1 = np.maximum(Z1, 0)
Z2 = A1 @ W2 + b2

print("Z1:", Z1.shape, "A1:", A1.shape, "Z2:", Z2.shape)
print("logits:", Z2.ravel().round(3))

**5. BCE punishes confident mistakes.** Cross-entropy explodes for confident WRONG answers: predicting 0.01 when the truth is 1 costs about 4.6.

In [ ]:
import numpy as np

def bce(y_true, p, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)
    return -np.mean(y_true * np.log(p) + (1 - y_true) * np.log(1 - p))

y = np.array([1.0, 0.0, 1.0])
p_close = np.array([0.9, 0.1, 0.8])
p_confident_wrong = np.array([0.01, 0.99, 0.8])

print(f"detector A (close)           : {bce(y, p_close):.3f}")
print(f"detector B (confidently wrong): {bce(y, p_confident_wrong):.3f}")

**6. Descend the bowl.** Measure the slope, step downhill, repeat - each step multiplies the error by 0.4 here.

In [ ]:
import numpy as np

def f(x):
    return (x - 3) ** 2

def g(x):
    return 2 * (x - 3)

lr, x = 0.3, 0.0
for step in range(1, 21):
    x -= lr * g(x)
    if step % 5 == 0:
        print(f"step {step:2d} | x = {x:+.5f} | f(x) = {f(x):.2e}")
print("final |x - 3|:", abs(x - 3))

**7. Goldilocks learning rate.** Convergence needs lr < 1 for this bowl: 0.01 crawls, 0.9 zig-zags inward, 1.1 grows the error 20% per step.

In [ ]:
import numpy as np

def g(x):
    return 2 * (x - 3)

for lr in [0.01, 0.9, 1.1]:
    x = 0.0
    for _ in range(15):
        x -= lr * g(x)
    print(f"lr={lr:<5} |x-3| after 15 steps = {abs(x - 3):.4f}")
# 0.01 -> too slow (barely moved); 0.9 -> converging zig-zag;
# 1.1 -> DIVERGED: each step multiplies the error by -1.2.

## Part 3 — Challenge

**8. Backprop by hand - then prove it.** The chain rule multiplies local sensitivities along the path; the numerical probe agrees to many decimals.

In [ ]:
import numpy as np

x, b, t, w = 1.0, 0.5, 1.0, 2.0

z = w * x + b                    # forward
a = 1 / (1 + np.exp(-z))
L = (a - t) ** 2
print(f"forward : z={z:.3f}  a={a:.4f}  L={L:.6f}")

dL_da = 2 * (a - t)              # backward, link by link
da_dz = a * (1 - a)
dz_dw = x
dz_db = 1.0
dL_dw = dL_da * da_dz * dz_dw
dL_db = dL_da * da_dz * dz_db
print(f"backward: dL/dw={dL_dw:+.6f}  dL/db={dL_db:+.6f}")

eps = 1e-6
def loss_with(weight):
    aa = 1 / (1 + np.exp(-(weight * x + b)))
    return (aa - t) ** 2
num = (loss_with(w + eps) - loss_with(w - eps)) / (2 * eps)
print(f"numerical dL/dw = {num:+.6f}  <- matches!")

**9. Why one neuron cannot solve XOR.** No single straight line carves XOR apart - 75% is the hard ceiling, which is why networks need LAYERS of non-linear units.

In [ ]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y = np.array([0, 1, 1, 0])

best = 0
for w1 in (-1, 1, 2):
    for w2 in (-1, 1, 2):
        for b in (-1.5, -0.5, 0.5, 1.5):
            p = (sigmoid(X @ np.array([w1, w2]) + b) > 0.5).astype(int)
            best = max(best, (p == y).mean())

print(f"Best XOR accuracy over 36 single neurons: {best:.0%}")